# ResNet20 Classification Demo

**Model:** ResNet20 trained on turntable dataset (saved weights)  
**Demo data:** Turntable test set — 645 unseen 96x96 grayscale sonar patches  
**Source:** `data_down/marine-debris-turntable-classification-object_classes-platform-96x96.hdf5` (x_test key)

In [ ]:
import numpy as np
import h5py
import os

# Suppress TensorFlow warnings
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import keras
import matplotlib.pyplot as plt

In [ ]:
# Load the trained ResNet20 model
model = keras.models.load_model('resnet20_turntable/model.keras')
print('Model loaded. Parameters:', model.count_params())

# Load test data — images the model has NEVER seen during training
with h5py.File('data_down/marine-debris-turntable-classification-object_classes-platform-96x96.hdf5', 'r') as f:
    x_test = f['x_test'][:]    # (645, 96, 96, 1) — grayscale sonar patches
    y_test = f['y_test'][:]    # (645,) — integer class labels
    class_names = [c.decode() if isinstance(c, bytes) else str(c) for c in f['class_names'][:]]

print('Test images:', x_test.shape)
print('Classes:', class_names)

In [ ]:
# Pick 10 random test images (different each run)
np.random.seed(None)
indices = np.random.choice(len(x_test), 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('ResNet20 Classification — Turntable Test Set (unseen data)', fontsize=14)

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    img = x_test[idx]
    true_label = int(y_test[idx])

    # Feed image through model → get probability for each of the 12 classes
    pred_probs = model.predict(img.reshape(1, 96, 96, 1), verbose=0)[0]

    # The class with the highest probability is the prediction
    pred_label = int(np.argmax(pred_probs))
    confidence = pred_probs[pred_label] * 100

    # Show the sonar patch with true vs predicted label
    ax.imshow(img.squeeze(), cmap='gray')
    true_name = class_names[true_label]
    pred_name = class_names[pred_label]

    # Green title = correct, Red title = wrong
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(
        'True: {}\nPred: {} ({:.0f}%)'.format(true_name, pred_name, confidence),
        fontsize=9, color=color, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.show()